In [1]:
!pip install yfinance

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [2]:
import sys
!{sys.executable} -m pip install yfinance tensorflow



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import random
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Dropout, Input, Flatten, MaxPooling1D, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers

# =====================================================================
# SEMILLA FIJA PARA REPRODUCIBILIDAD (Receta de Karpathy)
# =====================================================================
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

warnings.simplefilter(action="ignore", category=FutureWarning)

In [4]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS (Código del Profesor)
# =====================================================================
print("Descargando datos de Yahoo Finance...")
start_date = '1945-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

precios_close = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)['Close']
precios_close.dropna(axis=1, inplace=True)

# Cálculo de retornos logarítmicos
returns = np.log(precios_close).diff().dropna()
print(f"Forma de los datos de retornos: {returns.shape}")

# Función del profesor para crear ventanas
def create_time_series_data(data, input_window_size, output_window_size):
    X, y = [], []
    data_array = data.values if isinstance(data, pd.DataFrame) else data
    num_features = data_array.shape[1] 

    for i in range(len(data_array) - input_window_size - output_window_size + 1):
        input_sequence = data_array[i : i + input_window_size]
        X.append(input_sequence)
        
        if output_window_size > 0:
            output_sequence = data_array[i + input_window_size : i + input_window_size + output_window_size]
            average_output = np.mean(output_sequence, axis=0) 
            y.append(average_output)
        else:
            y.append(data_array[i + input_window_size - 1])
            
    return np.array(X), np.array(y)

Descargando datos de Yahoo Finance...
Forma de los datos de retornos: (16190, 23)


In [5]:
# =====================================================================
# 2. DEFINICIÓN DE ARQUITECTURA Y BASELINES
# =====================================================================

def construir_modelo_cnn(config, input_shape, n_assets=23):
    """Construye un modelo CNN 1D basado en la configuración dada."""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # L2 regularization según configuración
    l2_reg = regularizers.l2(config.get('l2', 0.0))
    
    # Primera capa convolucional
    model.add(Conv1D(filters=config['filters'], 
                     kernel_size=config['kernel_size'], 
                     activation='relu', 
                     padding='same',
                     kernel_regularizer=l2_reg))
    
    # Pooling según configuración
    if config.get('use_pooling', False):
        model.add(MaxPooling1D(pool_size=2))
    
    # Segunda capa convolucional (opcional)
    if config.get('double_conv', False):
        model.add(Conv1D(filters=config['filters']*2, 
                         kernel_size=config['kernel_size'], 
                         activation='relu', 
                         padding='same',
                         kernel_regularizer=l2_reg))
    
    # Aplanar o GlobalAveragePooling
    if config.get('use_flatten', False):
        model.add(Flatten())
    else:
        model.add(GlobalAveragePooling1D())
    
    model.add(Dropout(config['dropout']))
    model.add(Dense(n_assets, kernel_regularizer=l2_reg)) # Salida: 23 valores
    
    optimizador = Adam(learning_rate=config['lr'])
    model.compile(optimizer=optimizador, loss='mae')
    return model

def calcular_baselines(X_test, y_test, y_train_mean):
    """Calcula el MAE para modelos simples, incluyendo Buy and Hold."""
    
    # 1. Baseline Naive: El futuro será igual al último día de la ventana de entrada
    y_pred_naive = X_test[:, -1, :]
    mae_naive = np.mean(np.abs(y_pred_naive - y_test))
    
    # 2. Baseline SMA: El futuro será igual a la media de la ventana de entrada actual
    y_pred_sma = np.mean(X_test, axis=1)
    mae_sma = np.mean(np.abs(y_pred_sma - y_test))
    
    # 3. Baseline Buy and Hold: Predecir siempre la media histórica del entrenamiento
    # Creamos un array del mismo tamaño que y_test relleno con la media de y_train
    y_pred_bh = np.full_like(y_test, y_train_mean)
    mae_bh = np.mean(np.abs(y_pred_bh - y_test))
    
    return mae_naive, mae_sma, mae_bh

# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia', exist_ok=True)

In [6]:
# =====================================================================
# 3. CONFIGURACIÓN DEL EXPERIMENTO (ESTRATEGIA RNN ADAPTADA A CNN)
# =====================================================================

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

# =====================================================================
# BANCOS DE HIPERPARÁMETROS ESPECÍFICOS POR VENTANA (como RNN)
# =====================================================================

# Ventanas CORTAS de entrada (5 días) - Predicciones CORTAS (1, 5 días)
hp_in5_corto = [
    {'filters': 8, 'kernel_size': 3, 'dropout': 0.0, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 16, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 8, 'kernel_size': 3, 'dropout': 0.0, 'lr': 0.0001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
]

# Ventanas CORTAS de entrada (5 días) - Predicciones LARGAS (30, 90 días)
hp_in5_largo = [
    {'filters': 32, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.001, 'l2': 0.01, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.3, 'lr': 0.0005, 'l2': 0.01, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 32, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.005, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
]

# Ventanas MEDIAS de entrada (10 días) - Predicciones CORTAS (1, 5 días)
hp_in10_corto = [
    {'filters': 16, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 32, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.001, 'l2': 0.005, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 16, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.0005, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
]

# Ventanas MEDIAS de entrada (10 días) - Predicciones LARGAS (30, 90 días)
hp_in10_largo = [
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.0005, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.0, 'double_conv': True, 'use_pooling': False, 'use_flatten': False},
]

# Ventanas LARGAS de entrada (30 días) - Predicciones CORTAS (1, 5 días)
hp_in30_corto = [
    {'filters': 32, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
]

# Ventanas LARGAS de entrada (30 días) - Predicciones LARGAS (30, 90 días)
hp_in30_largo = [
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.0, 'double_conv': True, 'use_pooling': False, 'use_flatten': False},
    {'filters': 256, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.0, 'double_conv': True, 'use_pooling': False, 'use_flatten': False},
]

# Ventanas MUY LARGAS de entrada (90 días) - Predicciones CORTAS (1, 5 días)
hp_in90_corto = [
    {'filters': 16, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 32, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
]

# Ventanas MUY LARGAS de entrada (90 días) - Predicciones LARGAS (30, 90 días)
hp_in90_largo = [
    {'filters': 64, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.1, 'lr': 0.001, 'l2': 0.0, 'double_conv': False, 'use_pooling': False, 'use_flatten': False},
    {'filters': 128, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.0, 'double_conv': True, 'use_pooling': False, 'use_flatten': False},
    {'filters': 256, 'kernel_size': 3, 'dropout': 0.2, 'lr': 0.0005, 'l2': 0.0, 'double_conv': True, 'use_pooling': False, 'use_flatten': False},
]

# Matrices para reportar resultados finales (TRAIN, VAL, TEST)
matriz_mae_cnn_train = np.zeros((4, 4))
matriz_mae_cnn_val = np.zeros((4, 4))
matriz_mae_cnn_test = np.zeros((4, 4))
matriz_num_params = np.zeros((4, 4))
matriz_mae_naive = np.zeros((4, 4))
matriz_mae_sma = np.zeros((4, 4))
matriz_mae_bh = np.zeros((4, 4))

# Lista para guardar info detallada
resultados_detallados = []

# Early stopping con patience aumentado (como RNN pero con early stop activo)
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

In [7]:
# =====================================================================
# 4. BUCLE PRINCIPAL (AUTOMATIZACIÓN DE LOS 16 MODELOS x CONFIGURACIONES)
# =====================================================================

print("\nIniciando entrenamiento de modelos CNN...")

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):
        print(f"\n=======================================================")
        print(f" Ventana Entrada: {in_w} días | Ventana Salida: {out_w} días")
        print(f"=======================================================")
        
        # 1. Crear datos
        X, y = create_time_series_data(returns, in_w, out_w)
        
        # 2. Separación: 70% Train, 20% Validacion, 10% Test (como RNN)
        # IMPORTANTE: Mantenemos orden cronológico (sin shuffle) para series temporales
        split_1 = int(len(X) * 0.70)
        split_2 = int(len(X) * 0.90)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]

        # Calculamos la media global de entrenamiento para esta ventana (Buy and Hold)
        y_train_mean = np.mean(y_train, axis=0)
        
        # 3. Baselines
        mae_naive, mae_sma, mae_bh = calcular_baselines(X_test, y_test, y_train_mean)
        matriz_mae_naive[i, j] = mae_naive
        matriz_mae_sma[i, j] = mae_sma
        matriz_mae_bh[i, j] = mae_bh
        print(f"Baseline Naive (MAE en Test): {mae_naive:.6f}")
        print(f"Baseline SMA   (MAE en Test): {mae_sma:.6f}")
        print(f"Baseline Buy & Hold (MAE): {mae_bh:.6f}")

        # 4. Búsqueda del mejor modelo CNN (ESTRATEGIA RNN)
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None
        
        # SELECCIÓN DE BANCO DE HIPERPARÁMETROS (como RNN)
        if in_w == 5:
            lista_a_probar = hp_in5_corto if out_w in [1, 5] else hp_in5_largo
            nombre_lista = "In:5 Corto" if out_w in [1, 5] else "In:5 Largo"
        elif in_w == 10:
            lista_a_probar = hp_in10_corto if out_w in [1, 5] else hp_in10_largo
            nombre_lista = "In:10 Corto" if out_w in [1, 5] else "In:10 Largo"
        elif in_w == 30:
            lista_a_probar = hp_in30_corto if out_w in [1, 5] else hp_in30_largo
            nombre_lista = "In:30 Corto" if out_w in [1, 5] else "In:30 Largo"
        elif in_w == 90:
            lista_a_probar = hp_in90_corto if out_w in [1, 5] else hp_in90_largo
            nombre_lista = "In:90 Corto" if out_w in [1, 5] else "In:90 Largo"
        
        print(f" -> Usando banco de pruebas: [{nombre_lista}]")
        
        for config in lista_a_probar:
            print(f" -> Entrenando CNN: Filtros={config['filters']}, Kernel={config['kernel_size']}, LR={config['lr']}, Dropout={config['dropout']}, L2={config['l2']}...")
            
            modelo = construir_modelo_cnn(config, input_shape=(in_w, 23))
            
            # Usamos verbose=0 para no llenar la pantalla de números, epochs=50 es suficiente con EarlyStop
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   callbacks=[early_stop], 
                                   verbose=0)
            
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n[GANADOR] CNN con {mejor_config['filters']} filtros, kernel={mejor_config['kernel_size']}")
        
        # 5. Evaluación final del GANADOR en TRAIN, VAL y TEST
        mae_train_ganador = mejor_modelo.evaluate(X_train, y_train, verbose=0)
        mae_val_ganador = mejor_modelo.evaluate(X_val, y_val, verbose=0)
        mae_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        num_params = mejor_modelo.count_params()
        
        matriz_mae_cnn_train[i, j] = mae_train_ganador
        matriz_mae_cnn_val[i, j] = mae_val_ganador
        matriz_mae_cnn_test[i, j] = mae_test_ganador
        matriz_num_params[i, j] = num_params
        
        print(f"MAE en TRAIN: {mae_train_ganador:.6f}")
        print(f"MAE en VAL:   {mae_val_ganador:.6f}")
        print(f"MAE en TEST:  {mae_test_ganador:.6f}")
        print(f"Parámetros:   {num_params:,}")
        
        # Guardar info detallada
        resultados_detallados.append({
            'in_window': in_w,
            'out_window': out_w,
            'filters': mejor_config['filters'],
            'kernel_size': mejor_config['kernel_size'],
            'lr': mejor_config['lr'],
            'dropout': mejor_config['dropout'],
            'mae_train': mae_train_ganador,
            'mae_val': mae_val_ganador,
            'mae_test': mae_test_ganador,
            'num_params': num_params
        })
        
        # 6. Guardar Gráfica de Convergencia del Ganador
        plt.figure(figsize=(10, 5))
        plt.plot(mejor_historial.history['loss'], label='Error Entrenamiento (MAE)')
        plt.plot(mejor_historial.history['val_loss'], label='Error Validación (MAE)')
        
        # Título con todos los hiperparámetros
        filtros = mejor_config['filters']
        kernel = mejor_config['kernel_size']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        
        plt.title(f"Convergencia CNN | Filtros: {filtros} | Kernel: {kernel} | LR: {l_rate} | Drop: {d_out}\n(Ventana In:{in_w} - Out:{out_w})")
        
        plt.xlabel('Épocas')
        plt.ylabel('MAE')
        plt.legend()
        plt.grid(True)
        
        # Guardar la imagen
        nombre_archivo = f"graficas_convergencia/conver_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos CNN...

 Ventana Entrada: 5 días | Ventana Salida: 1 días
Baseline Naive (MAE en Test): 0.017789
Baseline SMA   (MAE en Test): 0.013608
Baseline Buy & Hold (MAE): 0.012243
 -> Usando banco de pruebas: [In:5 Corto]
 -> Entrenando CNN: Filtros=8, Kernel=3, LR=0.001, Dropout=0.0, L2=0.0...
 -> Entrenando CNN: Filtros=16, Kernel=3, LR=0.001, Dropout=0.1, L2=0.0...
 -> Entrenando CNN: Filtros=8, Kernel=3, LR=0.0001, Dropout=0.0, L2=0.0...

[GANADOR] CNN con 8 filtros, kernel=3
MAE en TRAIN: 0.011827
MAE en VAL:   0.010585
MAE en TEST:  0.012260
Parámetros:   767

 Ventana Entrada: 5 días | Ventana Salida: 5 días
Baseline Naive (MAE en Test): 0.013656
Baseline SMA   (MAE en Test): 0.008029
Baseline Buy & Hold (MAE): 0.005580
 -> Usando banco de pruebas: [In:5 Corto]
 -> Entrenando CNN: Filtros=8, Kernel=3, LR=0.001, Dropout=0.0, L2=0.0...
 -> Entrenando CNN: Filtros=16, Kernel=3, LR=0.001, Dropout=0.1, L2=0.0...
 -> Entrenando CNN: Filtros=8, Kernel=3, LR=

In [8]:
# =====================================================================
# 5. RESULTADOS FINALES (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*70)
print("MATRIZ DE RESULTADOS FINALES EN TEST (REDES CONVOLUCIONALES)")
print("="*70)
df_cnn_test = pd.DataFrame(matriz_mae_cnn_test, 
                           index=[f'In_{w}' for w in input_windows], 
                           columns=[f'Out_{w}' for w in output_windows])
print(df_cnn_test)

print("\n" + "="*70)
print("MATRIZ DE RESULTADOS EN TRAIN")
print("="*70)
df_cnn_train = pd.DataFrame(matriz_mae_cnn_train, 
                            index=[f'In_{w}' for w in input_windows], 
                            columns=[f'Out_{w}' for w in output_windows])
print(df_cnn_train)

print("\n" + "="*70)
print("MATRIZ DE RESULTADOS EN VALIDACIÓN")
print("="*70)
df_cnn_val = pd.DataFrame(matriz_mae_cnn_val, 
                          index=[f'In_{w}' for w in input_windows], 
                          columns=[f'Out_{w}' for w in output_windows])
print(df_cnn_val)

print("\n" + "="*70)
print("MATRIZ DE NÚMERO DE PARÁMETROS")
print("="*70)
df_params = pd.DataFrame(matriz_num_params, 
                         index=[f'In_{w}' for w in input_windows], 
                         columns=[f'Out_{w}' for w in output_windows])
print(df_params.astype(int))

print("\n" + "="*70)
print("BASELINES PARA COMPARACIÓN")
print("="*70)

print("\nNaive:")
df_naive = pd.DataFrame(matriz_mae_naive, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_naive)

print("\nSMA:")
df_sma = pd.DataFrame(matriz_mae_sma, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_sma)

print("\nBuy & Hold:")
df_bh = pd.DataFrame(matriz_mae_bh, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_bh)

# Tabla detallada con toda la información
print("\n" + "="*70)
print("TABLA DETALLADA DE RESULTADOS")
print("="*70)
df_detallado = pd.DataFrame(resultados_detallados)
print(df_detallado.to_string(index=False))



MATRIZ DE RESULTADOS FINALES EN TEST (REDES CONVOLUCIONALES)
          Out_1     Out_5    Out_30    Out_90
In_5   0.012260  0.005593  0.002329  0.001267
In_10  0.012290  0.005678  0.002336  0.001301
In_30  0.012293  0.005604  0.002315  0.001286
In_90  0.012280  0.005629  0.002332  0.001305

MATRIZ DE RESULTADOS EN TRAIN
          Out_1     Out_5    Out_30    Out_90
In_5   0.011827  0.005512  0.002226  0.001280
In_10  0.011895  0.005596  0.002183  0.001249
In_30  0.011874  0.005527  0.002145  0.001185
In_90  0.011862  0.005525  0.002153  0.001186

MATRIZ DE RESULTADOS EN VALIDACIÓN
          Out_1     Out_5    Out_30    Out_90
In_5   0.010585  0.004748  0.001950  0.001113
In_10  0.010615  0.004806  0.001973  0.001147
In_30  0.010602  0.004776  0.001965  0.001126
In_90  0.010604  0.004771  0.001973  0.001127

MATRIZ DE NÚMERO DE PARÁMETROS
       Out_1  Out_5  Out_30  Out_90
In_5     767    767    2999    2999
In_10   1511   1511  113431  113431
In_30   5975   5975  113431  113431
In_9

In [9]:
# =====================================================================
# 6. GRÁFICAS CONSOLIDADAS POR VENTANA DE SALIDA
# =====================================================================

print("\nGenerando gráficas consolidadas...")

for j, out_w in enumerate(output_windows):
    plt.figure(figsize=(10, 6))
    
    # Extraer resultados para esta ventana de salida
    resultados_out = []
    for i, in_w in enumerate(input_windows):
        resultados_out.append(matriz_mae_cnn_test[i, j])
    
    # Crear gráfica de barras
    x_pos = np.arange(len(input_windows))
    plt.bar(x_pos, resultados_out, alpha=0.7, color='steelblue')
    plt.xticks(x_pos, [f'In_{w}' for w in input_windows])
    plt.ylabel('MAE en Test')
    plt.xlabel('Ventana de Entrada')
    plt.title(f'Comparación de Modelos CNN para Ventana de Salida = {out_w} días')
    plt.grid(True, alpha=0.3)
    
    # Añadir valores encima de las barras
    for idx, val in enumerate(resultados_out):
        plt.text(idx, val, f'{val:.4f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig(f'graficas_convergencia/consolidada_out{out_w}.png')
    plt.close()

print("✅ Gráficas consolidadas guardadas en 'graficas_convergencia/'")


Generando gráficas consolidadas...
✅ Gráficas consolidadas guardadas en 'graficas_convergencia/'
